# 모델별 프롬프트 변형 슬롯 데모

CC `getAntModelOverrideSection` 백엔드 포팅 — `RenderContext.model` 슬롯 + `@[MODEL: <pattern>] ... @[/MODEL]` 마커 런타임 필터 + 어댑터 자동 model 주입.

## 작업순서
1. **Setup** — sys.path patch + import + registry snapshot/restore
2. **1부 베이스**: filter_model_blocks 단독 호출 (매칭/strip/None)
3. **2부 register/override**: 도메인 PromptSection 마커 박기 + render(ctx) 결과 확인
4. **3부 escape hatch + 위험 가드**: 어댑터 자동 주입 / 중첩 ValueError / 정적 hash 격리
5. **Cleanup** — registry restore
6. **다른 모듈 연계** — mock GeminiClient + AnthropicClient 두 어댑터 다른 model 자동 주입 + hash 격리
7. **실습 4개**

## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from best_agent_base.prompts.model_filter import filter_model_blocks
from best_agent_base.prompts.registry import registry
from best_agent_base.prompts.render import RenderContext, get_static_hash, render, _render_static

_REGISTRY_SNAPSHOT = dict(registry._sections)

def restore_state():
    registry._sections.clear()
    registry._sections.update(_REGISTRY_SNAPSHOT)
    print(f'restored — registry size={len(registry._sections)}')

print(f'베이스 7섹션: {sorted(registry._sections.keys())}')

## 1부: 베이스 — filter_model_blocks 단독

헬퍼 함수 자체의 동작 확인. 4 케이스: 매칭 keep / 미매칭 strip / model=None 전체 strip / 2 연속 블록.

In [ ]:
# 1.1 매칭 시 inner content keep
text = 'BASE @[MODEL: claude-*]CLAUDE-ONLY@[/MODEL]@[MODEL: gemini-*]GEMINI-ONLY@[/MODEL] TAIL'
print('claude:', repr(filter_model_blocks(text, 'claude-sonnet-4-5-20250929')))
print('gemini:', repr(filter_model_blocks(text, 'gemini-2.5-flash')))
print('none  :', repr(filter_model_blocks(text, None)))

In [ ]:
# 1.2 glob 패턴 정밀 매칭
text = '@[MODEL: claude-sonnet-4-*]sonnet-4@[/MODEL]@[MODEL: claude-opus-*]opus@[/MODEL]'
for m in ['claude-sonnet-4-5-20250929', 'claude-opus-4-7', 'gemini-2.5-flash']:
    print(f'{m:35s} → {filter_model_blocks(text, m)!r}')

In [ ]:
# 1.3 마커 없는 텍스트 — 그대로
print(repr(filter_model_blocks('plain text no markers', 'claude-anything')))

## 2부: register/override — 도메인 PromptSection 마커 박기

베이스 7섹션 중 Intro 를 도메인 버전으로 override — 모델별 다른 텍스트 박음.

In [ ]:
# 2.1 도메인 Intro override
class _MyIntro:
    name = 'Intro'
    static = True
    def render(self, ctx):
        return (
            'You are an agent.\n'
            '@[MODEL: claude-*]Use <thinking> tags for reasoning.@[/MODEL]'
            '@[MODEL: gemini-*]Use markdown ## headings for reasoning.@[/MODEL]'
        )

registry.register('Intro', _MyIntro())

for m in ['claude-sonnet-4-5-20250929', 'gemini-2.5-flash', None]:
    ctx = RenderContext(model=m)
    static_text = _render_static(ctx)
    # Intro 부분만 추출
    intro = static_text.split('\n\n')[0]
    print(f'model={m}:')
    print(intro)
    print()

In [ ]:
# 2.2 hash 격리 시연 — 같은 ctx shape, 다른 model → 다른 hash
h_claude = get_static_hash(RenderContext(model='claude-sonnet-4-5-20250929'))
h_gemini = get_static_hash(RenderContext(model='gemini-2.5-flash'))
h_none = get_static_hash(RenderContext())
print(f'claude: {h_claude}')
print(f'gemini: {h_gemini}')
print(f'none  : {h_none}')
print(f'모두 다름? {len({h_claude, h_gemini, h_none}) == 3}')

# 같은 모델 호출 시 동일 hash (캐시 적중)
h1 = get_static_hash(RenderContext(model='claude-sonnet-4-5-20250929'))
h2 = get_static_hash(RenderContext(model='claude-sonnet-4-5-20250929'))
print(f'\n같은 모델 반복 호출: {h1 == h2}')

## 3부: escape hatch + 위험 가드

어댑터 자동 주입 / 중첩 마커 거부 / model=None 조용한 정규화.

In [ ]:
# 3.1 중첩 마커 거부 (FR-6 / D5)
try:
    filter_model_blocks('@[MODEL: a]@[MODEL: b]X@[/MODEL]@[/MODEL]', 'a')
except ValueError as e:
    print(f'✅ ValueError 발화 — {e}')

In [ ]:
# 3.2 어댑터 자동 주입 시뮬레이션 (mock — 실제 API 호출 없음)
from unittest.mock import AsyncMock, MagicMock
import sys

# Mock SDK 등록
from best_agent_base.llm import gemini as gemini_mod
fake_sdk = MagicMock()
fake_result = MagicMock()
fake_result.text = 'ok'
fake_result.usage_metadata = MagicMock(prompt_token_count=1, candidates_token_count=1, cached_content_token_count=0)
fake_sdk.aio.models.generate_content = AsyncMock(return_value=fake_result)
fake_sdk.aio.caches.create = AsyncMock(side_effect=Exception('no cache for demo'))
_orig_build = gemini_mod._build_genai_client
gemini_mod._build_genai_client = lambda: fake_sdk

try:
    client = gemini_mod.GeminiClient()
    print(f'GeminiClient 의 model: {client._profile.model.value}')
    
    # ctx.model=None → 어댑터가 자동 주입
    captured = []
    _orig_split = gemini_mod.split_at_boundary
    def spy(ctx):
        captured.append(ctx.model)
        return _orig_split(ctx)
    gemini_mod.split_at_boundary = spy
    
    import asyncio
    asyncio.run(client.generate(RenderContext()))
    print(f'주입된 model: {captured[0]}')
    
    # ctx.model 명시 시 우회
    asyncio.run(client.generate(RenderContext(model='my-custom-model')))
    print(f'명시 우회 model: {captured[1]}')
    gemini_mod.split_at_boundary = _orig_split
finally:
    gemini_mod._build_genai_client = _orig_build

## Cleanup

In [ ]:
restore_state()

## 다른 모듈 연계

**시스템 프롬프트 정적/동적 분리** — BOUNDARY 위 정적부 + BOUNDARY 아래 동적부 양쪽에 filter 적용. **LLM 클라이언트 통합 (캐싱 흡수)** — Protocol 시그니처 무변경, 어댑터 자동 주입으로 KV 캐시가 모델별 독립. **어태치먼트 시스템** — `call_with_attachments` helper 가 `ctx.model` 그대로 전달.

## 실습 (4개)

### 실습 1 — 도메인 섹션 모델 분기
베이스 DoingTasks 섹션을 override 해서 claude-* 와 gemini-* 에 다른 작업 가이드 텍스트 박고, 두 ctx 호출 결과 비교.

### 실습 2 — 자체 catalog 정의
`MyDomainModel(StrEnum)` 같은 catalog 정의하고 `RenderContext(model=MyDomainModel.PRIMARY.value)` 호출.

### 실습 3 — 도구 description 시뮬레이션 (Phase 4 미리)
도구 description 문자열 안에 마커 박고 `filter_model_blocks` 단독 호출로 모델별 다른 description 추출 시연.

### 실습 4 — 캐시 격리 검증
3개 다른 모델 ctx 로 `get_static_hash` 호출 → 모두 다른 hash 확인. 그 다음 같은 모델로 5번 호출 → 모두 같은 hash 확인.

In [ ]:
# 실습 1 작성 영역
# class _MyDoingTasks:
#     ...


In [ ]:
# 실습 2 작성 영역
# from enum import StrEnum
# class MyDomainModel(StrEnum):
#     ...


In [ ]:
# 실습 3 작성 영역
# tool_desc = '도구 설명. @[MODEL: claude-*]...@[/MODEL] @[MODEL: gemini-*]...@[/MODEL]'


In [ ]:
# 실습 4 작성 영역
# hashes = [get_static_hash(RenderContext(model=m)) for m in ['claude-...', 'gemini-...', 'other']]
